# Feature Engineering Parque Solar Girasol

##### Cargar el DataFrame y convertir la columna de fecha a datetime

In [1]:
import pandas as pd

# Cargar el DataFrame desde el archivo Parquet
file_path = r"C:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\data\interim\meteo_data_with_generation\parque_solar_girasol.parquet"
df = pd.read_parquet(file_path)

# Mostrar las columnas actuales
print("Columnas antes del cambio:", df.columns.tolist())

# Si existe "date" y no existe "timestamp", renombrar "date" a "timestamp"
if "date" in df.columns and "timestamp" not in df.columns:
    df = df.rename(columns={"date": "timestamp"})

# Verificar que la columna ahora se llama "timestamp"
print("Columnas después del cambio:", df.columns.tolist())
df.head(5)

Columnas antes del cambio: ['date', 'generation', 'temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'apparent_temperature', 'surface_pressure', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'is_day', 'sunshine_duration', 'wet_bulb_temperature_2m', 'boundary_layer_height', 'shortwave_radiation', 'diffuse_radiation', 'global_tilted_irradiance', 'shortwave_radiation_instant', 'diffuse_radiation_instant', 'global_tilted_irradiance_instant', 'direct_radiation', 'direct_normal_irradiance', 'terrestrial_radiation', 'direct_radiation_instant', 'direct_normal_irradiance_instant', 'terrestrial_radiation_instant', 'pressure_msl']
Columnas después del cambio: ['timestamp', 'generation', 'temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'apparent_temperature', 'surface_pressure', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'et0_fao_evapotranspiration', '

,timestamp,generation,temperature_2m,dew_point_2m,relative_humidity_2m,apparent_temperature,surface_pressure,cloud_cover,cloud_cover_low,cloud_cover_mid,...,shortwave_radiation_instant,diffuse_radiation_instant,global_tilted_irradiance_instant,direct_radiation,direct_normal_irradiance,terrestrial_radiation,direct_radiation_instant,direct_normal_irradiance_instant,terrestrial_radiation_instant,pressure_msl
0,2021-07-09 00:00:00+00:00,1.85,27.088999,23.989000,83.185791,31.729336,1003.130005,25.0,10.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1013.099976
1,2021-09-06 21:00:00+00:00,0.00,27.239000,23.688999,80.982498,31.846409,1003.530884,17.0,13.0,12.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1013.500000
2,2021-09-06 22:00:00+00:00,0.00,27.139000,22.938999,77.852386,30.887413,1004.220764,24.0,18.0,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1014.200012
3,2021-09-06 23:00:00+00:00,0.00,26.739000,22.489000,77.558205,29.737156,1004.801575,17.0,11.0,12.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1014.799988
4,2021-09-07 00:00:00+00:00,0.00,26.338999,22.539000,79.650108,29.214397,1005.085449,12.0,9.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1015.099976


#### Eliminar registros problemáticos

In [2]:
import pandas as pd
import datetime

# Suponiendo que el DataFrame ya se cargó y se renombró la columna "date" a "timestamp"
# Obtener el mínimo y máximo de la serie de tiempo
tmin = df['timestamp'].min()
tmax = df['timestamp'].max()

# Filtrar registros: conservar solo aquellos entre tmin + 9 horas y tmax - 26 horas
df_filtered = df[(df['timestamp'] >= tmin + pd.Timedelta(hours=9)) & 
                 (df['timestamp'] <= tmax - pd.Timedelta(hours=26))]

print("Número de registros originales:", df.shape[0])
print("Número de registros después de filtrar:", df_filtered.shape[0])

Número de registros originales: 31288
Número de registros después de filtrar: 31261


#### Agregar características temporales

In [3]:
import numpy as np

def add_temporal_features(df):
    df = df.copy()
    # Extraer la hora del día
    df['hour'] = df['timestamp'].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Extraer el día de la semana
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    # Extraer el mes
    df['month'] = df['timestamp'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    return df

# Aplicar la función a df_filtered
df_temp = add_temporal_features(df_filtered)

# Mostrar las primeras filas con las nuevas características temporales
print("Características temporales agregadas:")
df_temp[['timestamp', 'hour', 'hour_sin', 'hour_cos', 'day_of_week', 'dow_sin', 'dow_cos', 'month', 'month_sin', 'month_cos']].head()

Características temporales agregadas:


,timestamp,hour,hour_sin,hour_cos,day_of_week,dow_sin,dow_cos,month,month_sin,month_cos
1,2021-09-06 21:00:00+00:00,21,-0.707107,0.707107,0,0.000000,1.00000,9,-1.0,-1.836970e-16
2,2021-09-06 22:00:00+00:00,22,-0.500000,0.866025,0,0.000000,1.00000,9,-1.0,-1.836970e-16
3,2021-09-06 23:00:00+00:00,23,-0.258819,0.965926,0,0.000000,1.00000,9,-1.0,-1.836970e-16
4,2021-09-07 00:00:00+00:00,0,0.000000,1.000000,1,0.781831,0.62349,9,-1.0,-1.836970e-16
5,2021-09-07 01:00:00+00:00,1,0.258819,0.965926,1,0.781831,0.62349,9,-1.0,-1.836970e-16


In [4]:
df_temp

,timestamp,generation,temperature_2m,dew_point_2m,relative_humidity_2m,apparent_temperature,surface_pressure,cloud_cover,cloud_cover_low,cloud_cover_mid,...,pressure_msl,hour,hour_sin,hour_cos,day_of_week,dow_sin,dow_cos,month,month_sin,month_cos
1,2021-09-06 21:00:00+00:00,0.00,27.239000,23.688999,80.982498,31.846409,1003.530884,17.0,13.0,12.0,...,1013.500000,21,-0.707107,7.071068e-01,0,0.000000,1.00000,9,-1.000000,-1.836970e-16
2,2021-09-06 22:00:00+00:00,0.00,27.139000,22.938999,77.852386,30.887413,1004.220764,24.0,18.0,18.0,...,1014.200012,22,-0.500000,8.660254e-01,0,0.000000,1.00000,9,-1.000000,-1.836970e-16
3,2021-09-06 23:00:00+00:00,0.00,26.739000,22.489000,77.558205,29.737156,1004.801575,17.0,11.0,12.0,...,1014.799988,23,-0.258819,9.659258e-01,0,0.000000,1.00000,9,-1.000000,-1.836970e-16
4,2021-09-07 00:00:00+00:00,0.00,26.338999,22.539000,79.650108,29.214397,1005.085449,12.0,9.0,6.0,...,1015.099976,0,0.000000,1.000000e+00,1,0.781831,0.62349,9,-1.000000,-1.836970e-16
5,2021-09-07 01:00:00+00:00,0.00,26.039000,22.338999,80.095230,28.825661,1004.778320,18.0,1.0,1.0,...,1014.799988,1,0.258819,9.659258e-01,1,0.781831,0.62349,9,-1.000000,-1.836970e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31257,2025-04-01 05:00:00+00:00,0.00,23.338999,18.739000,75.384201,24.746689,1005.182495,22.0,2.0,20.0,...,1015.299988,5,0.965926,2.588190e-01,1,0.781831,0.62349,4,0.866025,-5.000000e-01
31258,2025-04-01 06:00:00+00:00,0.00,23.188999,18.739000,76.070312,24.574907,1005.474426,22.0,10.0,14.0,...,1015.599976,6,1.000000,6.123234e-17,1,0.781831,0.62349,4,0.866025,-5.000000e-01
31259,2025-04-01 07:00:00+00:00,0.00,23.539000,18.838999,74.947006,25.049501,1006.179443,69.0,64.0,21.0,...,1016.299988,7,0.965926,-2.588190e-01,1,0.781831,0.62349,4,0.866025,-5.000000e-01
31260,2025-04-01 08:00:00+00:00,5.69,23.039000,20.088999,83.486206,24.923445,1007.152466,77.0,55.0,55.0,...,1017.299988,8,0.866025,-5.000000e-01,1,0.781831,0.62349,4,0.866025,-5.000000e-01


#### Agregar características de retardo (lags) y medias móviles

In [5]:
def add_lag_features(df, cols, lags=[1, 2, 3]):
    lag_series = []
    for col in cols:
        for lag in lags:
            s = df[col].shift(lag).rename(f"{col}_lag{lag}")
            lag_series.append(s)
    # Concatenar todas las series de lags en un único DataFrame y unir con el original
    df_lags = pd.concat(lag_series, axis=1)
    df = pd.concat([df, df_lags], axis=1)
    return df

def add_moving_average_features(df, cols, windows=[3, 6]):
    ma_series = []
    for col in cols:
        for w in windows:
            s = df[col].rolling(window=w, min_periods=1).mean().rename(f"{col}_ma{w}")
            ma_series.append(s)
    # Concatenar todas las series de medias móviles en un DataFrame y unir con el original
    df_ma = pd.concat(ma_series, axis=1)
    df = pd.concat([df, df_ma], axis=1)
    return df

# Seleccionar las columnas numéricas a partir de df_temp (puedes excluir 'generation' si es el target)
numeric_cols = df_temp.select_dtypes(include=[np.number]).columns.tolist()

# Aplicar funciones de ingeniería de características sin fragmentación
df_temp = add_lag_features(df_temp, numeric_cols, lags=[1, 2, 3])
df_temp = add_moving_average_features(df_temp, numeric_cols, windows=[3, 6])

print("Número total de columnas después de agregar lags y medias móviles:", df_temp.shape[1])

Número total de columnas después de agregar lags y medias móviles: 241


In [6]:
df_temp

,timestamp,generation,temperature_2m,dew_point_2m,relative_humidity_2m,apparent_temperature,surface_pressure,cloud_cover,cloud_cover_low,cloud_cover_mid,...,dow_sin_ma3,dow_sin_ma6,dow_cos_ma3,dow_cos_ma6,month_ma3,month_ma6,month_sin_ma3,month_sin_ma6,month_cos_ma3,month_cos_ma6
1,2021-09-06 21:00:00+00:00,0.00,27.239000,23.688999,80.982498,31.846409,1003.530884,17.0,13.0,12.0,...,0.000000,0.000000,1.000000,1.000000,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
2,2021-09-06 22:00:00+00:00,0.00,27.139000,22.938999,77.852386,30.887413,1004.220764,24.0,18.0,18.0,...,0.000000,0.000000,1.000000,1.000000,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
3,2021-09-06 23:00:00+00:00,0.00,26.739000,22.489000,77.558205,29.737156,1004.801575,17.0,11.0,12.0,...,0.000000,0.000000,1.000000,1.000000,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
4,2021-09-07 00:00:00+00:00,0.00,26.338999,22.539000,79.650108,29.214397,1005.085449,12.0,9.0,6.0,...,0.260610,0.195458,0.874497,0.905872,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
5,2021-09-07 01:00:00+00:00,0.00,26.039000,22.338999,80.095230,28.825661,1004.778320,18.0,1.0,1.0,...,0.521221,0.312733,0.748993,0.849396,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31257,2025-04-01 05:00:00+00:00,0.00,23.338999,18.739000,75.384201,24.746689,1005.182495,22.0,2.0,20.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01
31258,2025-04-01 06:00:00+00:00,0.00,23.188999,18.739000,76.070312,24.574907,1005.474426,22.0,10.0,14.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01
31259,2025-04-01 07:00:00+00:00,0.00,23.539000,18.838999,74.947006,25.049501,1006.179443,69.0,64.0,21.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01
31260,2025-04-01 08:00:00+00:00,5.69,23.039000,20.088999,83.486206,24.923445,1007.152466,77.0,55.0,55.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01


#### Eliminar filas con valores NaN generados por los lags/moving average

In [7]:
df_clean = df_temp.dropna()
print("Número de registros después de eliminar NaN:", df_clean.shape[0])


Número de registros después de eliminar NaN: 25495


In [8]:
df_clean

,timestamp,generation,temperature_2m,dew_point_2m,relative_humidity_2m,apparent_temperature,surface_pressure,cloud_cover,cloud_cover_low,cloud_cover_mid,...,dow_sin_ma3,dow_sin_ma6,dow_cos_ma3,dow_cos_ma6,month_ma3,month_ma6,month_sin_ma3,month_sin_ma6,month_cos_ma3,month_cos_ma6
4,2021-09-07 00:00:00+00:00,0.00,26.338999,22.539000,79.650108,29.214397,1005.085449,12.0,9.0,6.0,...,0.260610,0.195458,0.874497,0.905872,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
5,2021-09-07 01:00:00+00:00,0.00,26.039000,22.338999,80.095230,28.825661,1004.778320,18.0,1.0,1.0,...,0.521221,0.312733,0.748993,0.849396,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
6,2021-09-07 02:00:00+00:00,0.00,25.838999,21.938999,79.098320,28.400562,1004.078613,23.0,7.0,5.0,...,0.781831,0.390916,0.623490,0.811745,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
7,2021-09-07 03:00:00+00:00,0.00,25.838999,21.688999,77.899948,28.461506,1003.682556,26.0,15.0,22.0,...,0.781831,0.521221,0.623490,0.748993,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
8,2021-09-07 04:00:00+00:00,0.00,25.639000,21.938999,80.041924,28.384056,1003.675964,25.0,16.0,23.0,...,0.781831,0.651526,0.623490,0.686242,9.0,9.0,-1.000000,-1.000000,-1.836970e-16,-1.836970e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31257,2025-04-01 05:00:00+00:00,0.00,23.338999,18.739000,75.384201,24.746689,1005.182495,22.0,2.0,20.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01
31258,2025-04-01 06:00:00+00:00,0.00,23.188999,18.739000,76.070312,24.574907,1005.474426,22.0,10.0,14.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01
31259,2025-04-01 07:00:00+00:00,0.00,23.539000,18.838999,74.947006,25.049501,1006.179443,69.0,64.0,21.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01
31260,2025-04-01 08:00:00+00:00,5.69,23.039000,20.088999,83.486206,24.923445,1007.152466,77.0,55.0,55.0,...,0.781831,0.781831,0.623490,0.623490,4.0,4.0,0.866025,0.866025,-5.000000e-01,-5.000000e-01


#### Guardar el DataFrame procesado

In [10]:
import os

# Definir la carpeta de salida
output_folder = r"../data/interim/meteo_data_with_generation_clean"
os.makedirs(output_folder, exist_ok=True)

# Definir el nombre del archivo
output_filename = "parque_solar_girasol_clean.parquet"
output_path = os.path.join(output_folder, output_filename)

# Guardar el DataFrame
df_clean.to_parquet(output_path, index=False)
print(f"Archivo guardado en: {output_path}")


Archivo guardado en: ../data/interim/meteo_data_with_generation_clean\parque_solar_girasol_clean.parquet
